# Label-shuffle leakage test


**Important:** Place `merged_physics.csv` and `uav_strict_utils.py` in the same folder as this notebook before running.  
The notebook saves `metrics.csv`, `classification_report.txt`, `predictions.csv`, `confusion_matrix.pdf`, `metrics_bar.pdf`, `training_curve.pdf`, and the trained `.keras` model for each technique.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from uav_strict_utils import *

INPUT_FILE = 'merged_physics.csv'
RESULT_ROOT = Path('Label_Shuffle_Test_Results')
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 50
STRIDE = 50
EPOCHS = 80
BATCH_SIZE = 16

_, X, y, y_text, groups, label_encoder, feature_cols = prepare_windows(
    INPUT_FILE, window_size=WINDOW_SIZE, stride=STRIDE, filter_active=True
)

file_labels = pd.DataFrame({'source_file': groups, 'label': y_text}).drop_duplicates()
file_majority = file_labels.groupby('source_file')['label'].agg(lambda s: s.mode()[0]).reset_index()
files = file_majority['source_file'].values
file_y = file_majority['label'].values

trainval_files, test_files, trainval_labels, test_labels = train_test_split(
    files, file_y, test_size=0.33, stratify=file_y, random_state=SEED
)
try:
    train_files, val_files, train_labels, val_labels = train_test_split(
        trainval_files, trainval_labels, test_size=0.33, stratify=trainval_labels, random_state=SEED
    )
except ValueError:
    train_files, val_files, train_labels, val_labels = train_test_split(
        trainval_files, trainval_labels, test_size=0.33, random_state=SEED
    )

train_idx = np.where(np.isin(groups, train_files))[0]
val_idx = np.where(np.isin(groups, val_files))[0]
test_idx = np.where(np.isin(groups, test_files))[0]

X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
X_train, X_val, X_test, scaler = scale_by_train(X_train, X_val, X_test)

chance_level = 1.0 / len(label_encoder.classes_)
print('Expected chance level:', chance_level)
print('If shuffled-label accuracy is still high, there is likely leakage.')

all_metrics = []
for model_name in MODEL_NAMES:
    metrics = train_one_model(
        model_name, X_train, y_train, X_val, y_val, X_test, y_test,
        label_encoder, feature_cols, RESULT_ROOT / model_name,
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle_labels=True
    )
    metrics['Model'] = model_name
    metrics['Expected_Chance_Level'] = chance_level
    all_metrics.append(metrics)

summary_df = pd.DataFrame(all_metrics)
summary_df = summary_df[['Model', 'Expected_Chance_Level'] + [c for c in summary_df.columns if c not in ['Model', 'Expected_Chance_Level']]]
summary_df.to_csv(RESULT_ROOT / 'label_shuffle_test_summary.csv', index=False)
summary_df


C:\Users\Hp\miniconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Idle rows removed: 47
Dataset shape after cleaning/filtering: (1396, 33)
Number of numerical features: 31
Class counts after filtering:
class_label
bend      591
normal    589
crack     216
Name: count, dtype: int64

Created non-overlapping windows
Total windows: 21
Window class counts:
bend      9
normal    9
crack     3
Name: count, dtype: int64
Source files: 9
Expected chance level: 0.3333333333333333
If shuffled-label accuracy is still high, there is likely leakage.

Epoch 1/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.4000 - loss: 1.0957 - val_accuracy: 0.5000 - val_loss: 1.0328 - learning_rate: 0.0010
Epoch 2/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 785ms/step - accuracy: 0.6000 - loss: 0.8452 - val_accuracy: 0.5000 - val_loss: 1.0215 - learning_rate: 0.0010
Epoch 3/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step - accuracy: 0.8000 - loss: 0.6732 - val_accuracy: 0.5000 - val_loss: 1.0010 - learning_rate: 0.0010
Epoch 4/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step - accuracy: 0.7000 - loss: 

,Model,Expected_Chance_Level,Accuracy,Precision,Recall,F1Score,Precision_weighted,Recall_weighted,F1Score_weighted
0,1D_CNN,0.333333,0.428571,0.300000,0.333333,0.300000,0.385714,0.428571,0.385714
1,LSTM,0.333333,0.285714,0.111111,0.222222,0.148148,0.142857,0.285714,0.190476
2,CNN_LSTM,0.333333,0.571429,0.500000,0.444444,0.388889,0.642857,0.571429,0.500000
3,TCN,0.333333,0.428571,0.277778,0.333333,0.301587,0.357143,0.428571,0.387755
4,Transformer,0.333333,0.428571,0.277778,0.333333,0.301587,0.357143,0.428571,0.387755
5,Multimodal_Transformer,0.333333,0.571429,0.366667,0.666667,0.472222,0.328571,0.571429,0.416667
